# Acemi Kaggle runner — Stage 4 refinement

Clones the public repository, installs Stockfish and tools, runs the correctness suite, then two matches, and pushes only `data/` records. No analysis. Keep Internet enabled.

1. SPRT: current engine (0.4.1) against the previous version (0.4.0, commit `c62fc27`) at 200 ms per move.
2. First rough Elo anchor: 20 games against Stockfish limited to UCI_Elo 1320 at 500 ms per move (a rough scale, not comparable with Lichess or CCRL ratings).

Before running: create a Kaggle Secret named `GITHUB_TOKEN`. Never paste the token into a cell.


In [ ]:
GITHUB_REPO_URL = "https://github.com/xdxddxdxd/acemi.git"
GITHUB_BRANCH = "main"
WORKSPACE = "/kaggle/working/acemi"


In [ ]:
import os
import shutil
import subprocess
from pathlib import Path

def run_quietly(args, *, cwd=None):
    result = subprocess.run(args, cwd=cwd, text=True, capture_output=True)
    if result.returncode:
        details = "\n".join(part for part in (result.stdout, result.stderr) if part)
        raise RuntimeError(f"Command failed: {' '.join(args)}\n{details[-4000:]}")
    return result.stdout

repo_url = GITHUB_REPO_URL.strip()
if not repo_url.startswith("https://github.com/") or "YOUR_ACCOUNT" in repo_url:
    raise ValueError("Set GITHUB_REPO_URL to your repository's HTTPS clone URL.")
if "@" in repo_url.split("github.com/", 1)[-1].split("/", 1)[0]:
    raise ValueError("Do not put credentials in GITHUB_REPO_URL. Use the GITHUB_TOKEN Kaggle Secret.")
workspace = Path(WORKSPACE)
if not workspace.exists():
    run_quietly(["git", "clone", "--branch", GITHUB_BRANCH, repo_url, str(workspace)])
else:
    run_quietly(["git", "-C", str(workspace), "pull", "--ff-only", "origin", GITHUB_BRANCH])
os.chdir(workspace)
print(f"Repository ready on branch {GITHUB_BRANCH}.")


In [ ]:
run_quietly(["apt-get", "update", "-qq"])
run_quietly(["apt-get", "install", "-y", "-qq", "stockfish"])
run_quietly(["python", "-m", "pip", "install", "-q", "-r", "requirements-tools.txt"])
stockfish_bin = shutil.which("stockfish") or ("/usr/games/stockfish" if Path("/usr/games/stockfish").exists() else None)
if stockfish_bin is None:
    raise RuntimeError("Stockfish package installed but executable was not found.")
version = subprocess.run([stockfish_bin, "--version"], check=True, text=True, capture_output=True).stdout.splitlines()[0]
print(f"Stockfish installed: {version}")
print("Tools installed.")


In [ ]:
# Correctness suite: board, perft, UCI, stats, eval, search.
test_result = subprocess.run(["python", "-m", "unittest", "discover", "-v"], text=True, capture_output=True)
if test_result.stdout:
    print(test_result.stdout, end="")
if test_result.stderr:
    print(test_result.stderr, end="")
if test_result.returncode:
    raise RuntimeError("Correctness suite failed; see test output above.")
print("Correctness suite passed.")


In [ ]:
# SPRT: current engine vs the previous version (0.4.0, commit c62fc27), elo0=0 elo1=10, 200 ms per move.
# SPRT stops early on a clear result; 300 games is the cap (undecided = inconclusive).
OLD_COMMIT = "c62fc27"
old_dir = Path("/kaggle/working/acemi_v040")
if not old_dir.exists():
    run_quietly(["git", "clone", repo_url, str(old_dir)])
    run_quietly(["git", "-C", str(old_dir), "checkout", OLD_COMMIT])
subprocess.run(
    [
        "python", "-m", "tools.match",
        "--games", "300",
        "--movetime-ms", "200",
        "--opponent-cmd", "python -u -m engine",
        "--opponent-cwd", str(old_dir),
        "--sprt", "--elo0", "0", "--elo1", "10",
    ],
    check=True,
)


In [ ]:
# Rough Elo anchor: Acemi vs Stockfish with UCI_LimitStrength, UCI_Elo 1320, 500 ms per move.
subprocess.run(
    [
        "python", "-m", "tools.match",
        "--games", "20",
        "--movetime-ms", "500",
        "--uci-elo", "1320",
    ],
    check=True,
)


In [ ]:
import base64
from kaggle_secrets import UserSecretsClient

try:
    github_token = UserSecretsClient().get_secret("GITHUB_TOKEN")
except Exception:
    raise RuntimeError("Kaggle Secret GITHUB_TOKEN is unavailable; add it under Add-ons → Secrets.") from None
if not github_token:
    raise RuntimeError("Kaggle Secret GITHUB_TOKEN is empty.")
auth_value = "AUTHORIZATION: basic " + base64.b64encode(f"x-access-token:{github_token}".encode()).decode()
git_env = os.environ.copy()
git_env.update({
    "GIT_CONFIG_COUNT": "1",
    "GIT_CONFIG_KEY_0": "http.https://github.com/.extraheader",
    "GIT_CONFIG_VALUE_0": auth_value,
    "GIT_TERMINAL_PROMPT": "0",
})
def git(*args):
    result = subprocess.run(["git", *args], cwd=workspace, env=git_env, text=True, capture_output=True)
    safe_error = result.stderr.replace(github_token, "[REDACTED]")
    if result.returncode:
        raise RuntimeError(f"Git command failed ({args[0]}): {safe_error[-1500:]}")
    return result.stdout
git("config", "user.name", "Acemi Kaggle Runner")
git("config", "user.email", "acemi-kaggle-runner@users.noreply.github.com")
git("add", "--", "data/runs", "data/index.csv", "data/positions")
if git("status", "--porcelain").strip():
    git("commit", "-m", "data: record Stage 4 refinement (vs 0.4.0, vs Stockfish 1320)")
    git("push", "origin", f"HEAD:{GITHUB_BRANCH}")
    print("Run records pushed to GitHub.")
else:
    print("No new run records to push.")
del github_token, auth_value, git_env
